# CrewAI Multi-Agent Crew - Research & Writeup

This notebook shows how to build a **CrewAI crew**: a team of role-based agents that collaborate on a research and writing task.

We will:
1. Install CrewAI and the OpenAI integration.
2. Configure the OpenAI API key.
3. Create two agents — a **Researcher** and a **Writer**.
4. Assign tasks and run the crew sequentially.


## Section 1: Install Dependencies


**Install the CrewAI framework.**

- `crewai` is the core library for building role-based multi-agent crews.
- Run this cell and restart the kernel if this is the first time installing.


In [1]:
!pip install crewai

Defaulting to user installation because normal site-packages is not writeable
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_multipart-0.0.29-py3-none-any.whl.metadata (2.1 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


**Install the LangChain OpenAI integration.**

- `langchain-openai` provides `ChatOpenAI`, which CrewAI can use as its LLM backend.
- This lets us instantiate an explicit LLM object with a chosen model and API key.


In [ ]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


## Section 2: Environment Setup


**Load environment variables and set the OpenAI API key.**

What the code does, line by line:
- The commented lines show how to install all optional packages at once if needed.
- `load_dotenv(override=True)` loads variables from a local `.env` file.
- On Google Colab, `userdata.get('OPENAI_API_KEY')` reads a secret stored in the notebook environment.
- `os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY` makes the key available to CrewAI and OpenAI models.
- If no key is found, a tip is printed and the crew setup still runs, but `kickoff()` is skipped.


In [ ]:
# Install (uncomment if needed). Run this cell, then restart the kernel.
# Note: package names are intentionally explicit to avoid version confusion.
# !pip install -U langchain langchain-openai langchain-community crewai pyautogen chromadb

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Read API keys if available (optional for sections that use OpenAI)
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if not OPENAI_API_KEY:
    print("Tip: Set OPENAI_API_KEY in your environment if you want to use OpenAI-backed sections.")

## Section 3: Build the CrewAI Crew


**Create agents, tasks, and a sequential crew.**

What the code does, line by line:
- Tries to import `Agent`, `Task`, `Crew`, and `Process` from `crewai`, plus `ChatOpenAI` from `langchain_openai`.
- If an API key is available, creates an explicit `llm = ChatOpenAI(...)`. Otherwise `llm` stays `None`.
- Defines the **Researcher** agent with a role, goal, backstory, and `verbose=True` so you can see its reasoning.
- Defines the **Writer** agent, whose job is to turn research into a summary and comparison table.
- Creates `task_research`: gather pros, cons, and use-cases for GPT-4, Claude, and Mistral.
- Creates `task_write`: produce a one-paragraph summary and a Markdown comparison table.
- Builds the `Crew` with both agents, both tasks, and `Process.sequential` so the writer waits for the researcher.
- If an LLM is available, calls `crew.kickoff()` and prints the first 1200 characters of the result.
- If a package is missing, the block prints a friendly skip message instead of crashing.

Expected output:
- Verbose logs from each agent show the tasks being executed.
- The final output is a structured comparison of GPT-4, Claude, and Mistral.


In [ ]:
try:
    from crewai import Agent, Task, Crew, Process
    # Optional LLM via LangChain (CrewAI can also use other providers)
    from langchain_openai import ChatOpenAI

    llm = None
    if OPENAI_API_KEY:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key = OPENAI_API_KEY)

    else:
        print("OPENAI_API_KEY not set — CrewAI will run with no LLM (agents will be configured but kickoff may be limited).")

    researcher = Agent(
        role="Researcher",
        goal="Analyze LLMs and extract pros/cons for GPT-4, Claude, and Mistral.",
        backstory="You specialize in balanced evaluations of foundation models.",
        verbose=True,
        llm="gpt-4o-mini"
    )

    writer = Agent(
        role="Writer",
        goal="Summarize findings into a concise comparison table and bullet list.",
        backstory="You convert technical findings into clear outputs for executives.",
        verbose=True,
        llm="gpt-4o-mini"
    )

    task_research = Task(
        description="Research the strengths, weaknesses, and ideal use-cases of GPT-4, Claude, and Mistral. Return ~6 bullets per model.",
        agent=researcher,
        expected_output="A structured bullet list per model with at least one risk/limitation called out."
    )

    task_write = Task(
        description="Turn the research into a one-paragraph summary + a compact table with at least 4 comparison dimensions.",
        agent=writer,
        expected_output="One paragraph + a Markdown table. Keep it neutral and cite sources if any were used."
    )

    crew = Crew(
        agents=[researcher, writer],
        tasks=[task_research, task_write],
        process=Process.sequential
    )

    print("CrewAI: constructed crew with 2 agents and 2 tasks.")
    # Running kickoff requires an LLM; if no key is set, we skip actual execution.
    if llm:
        result = crew.kickoff()
        print("\n=== CrewAI result (truncated) ===\n")
        print(str(result)[:1200])
    else:
        print("Skipped kickoff because OPENAI_API_KEY was not provided.")
except ModuleNotFoundError as e:
    print("CrewAI section skipped. Missing package:", e.name)
    print("Install with: pip install crewai langchain-openai")

CrewAI: constructed crew with 2 agents and 2 tasks.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Research the strengths, weaknesses, and ideal use-cases of GPT-4, Claude, and Mistral. Return ~6         │
│  bullets per model.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### GPT-4                                                                                                      │
│                                                                                                                 │
│  - **Strengths:**                                                                                               │
│    - **High Comprehension and Context Retention:** Excels in understanding complex prompts and maintaining      │
│  context over longer interactions.                                                                              │
│    - **Versatile Application:** Suitable for a wide range of tasks, including creative writing, coding          │
│  support, and data analysis.                                                                                    │
│    - **Multimodal Capabilities:** Can process and generate text-based responses alongside image inputs,         │
│  enhancing its utility across various domains.                                                                  │
│    - **Fine-Tuning and Customization:** Offers fine-tuning capabilities, allowing specific adaptations for      │
│  certain industries or tasks, increasing relevance and accuracy.                                                │
│    - **Robust API Access:** Accessible through an API, facilitating integration into various applications       │
│  seamlessly.                                                                                                    │
│    - **Enhanced Safety Features:** Incorporated safety measures to minimize harmful outputs and improve user    │
│  experience.                                                                                                    │
│                                                                                                                 │
│  - **Weaknesses:**                                                                                              │
│    - **Resource-Intensive:** Requires significant computational resources for both training and deployment,     │
│  which can be a barrier to entry for some users.                                                                │
│    - **Occasional Inaccuracies:** May produce confidently incorrect information or "hallucinations" where it    │
│  generates plausible but false statements.                                                                      │
│    - **Limited Real-World Knowledge:** Knowledge base is static and only extends to information available up    │
│  to 2023.                                                                                                       │
│                                                                                                                 │
│  - **Ideal Use Cases:**                                                                                         │
│    - **Creative Writing and Storytelling:** Great for generating novels, poetry, and other creative content.    │
│    - **Coding Assistance:** Valuable tool for developers seeking help with code snippets and debugging.         │
│    - **Research and Information Gathering:** Useful for summarizing research papers and collating information   │
│  on complex topics.                                                                                             │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Task: Turn the research into a one-paragraph summary + a compact table with at least 4 comparison dimensions.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Summary                                                                                                    │
│                                                                                                                 │
│  The comparison between GPT-4, Claude, and Mistral highlights their respective strengths and weaknesses across  │
│  various dimensions, providing valuable insights for organizations considering implementation. GPT-4 excels in  │
│  versatile applications and multimodal capabilities but requires significant resources. Claude emphasizes       │
│  conversational clarity and ethical alignment, ideal for customer-centric applications but has narrower         │
│  training data. In contrast, Mistral shines in efficiency and cost-effectiveness, appealing to startups and     │
│  niche applications, though it may lack the knowledge depth of its competitors. Each model presents unique use  │
│  cases, risks, and limitations that stakeholders should consider based on their specific needs.                 │
│                                                                                                                 │
│  | Dimension              | GPT-4                                   | Claude                                    │
│  | Mistral                                 |                                                                    │
│  |-----------------------|-----------------------------------------|----------------------------------------|-  │
│  ----------------------------------------|                                                                      │
│  | **Strengths**         | High comprehension, versatile, multimodal | Conversational clarity, ethical focus    │
│  | Efficient, open-source, customizable     |                                                                   │
│  | **Weaknesses**        | Resource-intensive, inaccuracies        | Limited training data, integration lag |   │
│  Limited scope, community-driven updates  |                                                                     │
│  | **Ideal Use Cases**   | Creative writing, coding assistance     | Customer support, education            |   │
│  Startups, research, localized apps      |                                                                      │
│  | **Risks/Limitation**  | Over-reliance on outputs               | Inconsistent across domains            |    │
│  Reliance on community for updates        |                                                                     │
│                                                                                                                 │
│  Sources: Original research findings on GPT-4, Claude, and Mistral capabilities.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== CrewAI result (truncated) ===

### Summary

The comparison between GPT-4, Claude, and Mistral highlights their respective strengths and weaknesses across various dimensions, providing valuable insights for organizations considering implementation. GPT-4 excels in versatile applications and multimodal capabilities but requires significant resources. Claude emphasizes conversational clarity and ethical alignment, ideal for customer-centric applications but has narrower training data. In contrast, Mistral shines in efficiency and cost-effectiveness, appealing to startups and niche applications, though it may lack the knowledge depth of its competitors. Each model presents unique use cases, risks, and limitations that stakeholders should consider based on their specific needs.

| Dimension              | GPT-4                                   | Claude                                  | Mistral                                 |
|-----------------------|--------------------------------

---

**Summary**

In this notebook you built a CrewAI crew that:
- Uses role-based agents (`Researcher` and `Writer`).
- Runs tasks sequentially (`Process.sequential`).
- Gracefully skips execution if the OpenAI API key or packages are missing.

You can extend this pattern by adding more agents, more tasks, or a hierarchical process.
